# Imports

In [1]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, pointbiserialr
from scipy.stats import f_oneway, kruskal

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, roc_curve, auc

from autogluon.tabular import TabularDataset, TabularPredictor
#from autogluon.multimodal import MultiModalPredictor
import xgboost as xgb

plt.style.use('default')
sns.set_palette("husl")
SEED = 43
np.random.seed(SEED)

SCORING = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error',
    'mse': 'neg_mean_squared_error'
}
GRID_SEARCH_KWARGS = dict(
    cv=10,
    scoring=SCORING,
    refit='r2',
    n_jobs=-1,
    return_train_score=True
)
LINEAR_PARAM_GRID = {
    'model__fit_intercept': [True, False],
    'model__positive': [False, True]
}
TREE_PARAM_GRID = {
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
RF_PARAM_GRID = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [1, 100],
    'model__min_samples_leaf': [2, 100],
    'model__max_features': [None, 'sqrt', 'log2']
}
MLP_PARAM_GRID = {
    'model__hidden_layer_sizes': [(100,), (100, 50), (50, 25)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [0.0001, 0.001, 0.01],
    'model__learning_rate_init': [0.001, 0.01]
}
XGB_PARAM_GRID = {
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__n_estimators': [50, 100, 200]
}

def evaluate_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    errors = y_true - y_pred

    bin_size = 0.1
    bins = np.arange(0.0, 1.0 + bin_size, bin_size)
    bin_ids = np.digitize(y_pred, bins, right=False)

    ece = 0.0
    N = len(y_pred)

    for i in range(1, len(bins)):
        mask = bin_ids == i
        if not np.any(mask):
            continue

        bin_center = (bins[i - 1] + bins[i]) / 2.0
        median_obs = np.median(y_true[mask])
        ece += (mask.sum() / N) * abs(median_obs - bin_center)
        
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': float(np.mean(np.abs(errors))),
        'mse': float(np.mean(errors ** 2)),
        "p95": np.quantile(np.abs(errors), 0.95),
        'ece': float(ece),              # Expected calibration error
    }


def record_regression_results(results, target, model_name, y_true, y_pred, split='test'):
    metrics = evaluate_regression_metrics(y_true, y_pred)
    split_label = split.capitalize()
    print(f"{split_label} R2 score {metrics['r2']:.4f}")
    print(f"{split_label} MAE score {metrics['mae']:.4f}")
    print(f"{split_label} MSE score {metrics['mse']:.4f}")
    print(f"{split_label} P95 score {metrics['p95']:.4f}")
    print(f"{split_label} ECE score {metrics['ece']:.4f}")
    results.append({
        'target': target,
        'model': model_name,
        f'{split}_r2': metrics['r2'],
        f'{split}_mae': metrics['mae'],
        #f'{split}_mse': metrics['mse'],
        #f'{split}_p95': metrics['p95'],
        f'{split}_ece': metrics['ece'],
    })
    return metrics


# Load data

In [2]:
model_det = "yolo"
data_path_file = f"{model_det}_metadetect.csv"
data = pd.read_csv("../data/" + data_path_file, index_col=0)

print(data)

          num_detections  md_N_min  md_N_max  md_N_mean  md_N_std  \
image_id                                                            
1                 1376.0       1.0      15.0   1.453488  1.562418   
6                 1563.0       1.0      18.0   1.279590  1.074576   
39                1635.0       1.0      19.0   1.223242  1.110835   
49                1617.0       1.0      16.0   1.236858  0.972816   
52                1339.0       1.0      33.0   1.493652  1.643939   
...                  ...       ...       ...        ...       ...   
99904             1453.0       1.0      19.0   1.376462  1.436204   
99939             1876.0       1.0      15.0   1.066098  0.548211   
99941             1524.0       1.0      16.0   1.312336  1.331855   
99984             1755.0       1.0      12.0   1.139601  0.621916   
99997             1726.0       1.0      16.0   1.158749  0.833991   

          md_rmin_min  md_rmin_max  md_rmin_mean  md_rmin_std  md_rmax_min  \
image_id                

In [3]:
data = data.dropna()
num_rows, num_cols = data.shape
print(f"After dropping missing values: {num_rows:,} rows and {num_cols} columns")

After dropping missing values: 9,667 rows and 193 columns


In [4]:
iou = data["iou"]
lrp = data["lrp"]
conf = data["md_s_mean"]
image_paths_dict = {int(img_pth.split("_")[0]):f"../data/zod_yolo/images/test/{img_pth}" for img_pth in os.listdir("../data/zod_yolo/images/test") if img_pth.endswith(".jpg")}
img_path = pd.DataFrame.from_dict(image_paths_dict, orient='index', columns=["Images"])

data = data.drop(columns=["iou", "lrp"])
data_indices = data.index.to_numpy()

In [5]:
all_columns = data.columns    

print(f"Number of rows: {data.shape[0]}")
print(f"Columns: {all_columns.sort_values()}")
print(f"Number of columns: {len(all_columns)}")

Number of rows: 9667
Columns: Index(['md_N_max', 'md_N_mean', 'md_N_min', 'md_N_std', 'md_cmax_cand_max_max',
       'md_cmax_cand_max_mean', 'md_cmax_cand_max_min', 'md_cmax_cand_max_std',
       'md_cmax_cand_mean_max', 'md_cmax_cand_mean_mean',
       ...
       'md_s_cand_min_std', 'md_s_cand_std_max', 'md_s_cand_std_mean',
       'md_s_cand_std_min', 'md_s_cand_std_std', 'md_s_max', 'md_s_mean',
       'md_s_min', 'md_s_std', 'num_detections'],
      dtype='object', length=191)
Number of columns: 191


In [6]:
numeric_columns = all_columns
print(sorted(numeric_columns))
assert len(all_columns) == len(numeric_columns), "Columns not match"

['md_N_max', 'md_N_mean', 'md_N_min', 'md_N_std', 'md_cmax_cand_max_max', 'md_cmax_cand_max_mean', 'md_cmax_cand_max_min', 'md_cmax_cand_max_std', 'md_cmax_cand_mean_max', 'md_cmax_cand_mean_mean', 'md_cmax_cand_mean_min', 'md_cmax_cand_mean_std', 'md_cmax_cand_min_max', 'md_cmax_cand_min_mean', 'md_cmax_cand_min_min', 'md_cmax_cand_min_std', 'md_cmax_cand_std_max', 'md_cmax_cand_std_mean', 'md_cmax_cand_std_min', 'md_cmax_cand_std_std', 'md_cmax_max', 'md_cmax_mean', 'md_cmax_min', 'md_cmax_std', 'md_cmin_cand_max_max', 'md_cmin_cand_max_mean', 'md_cmin_cand_max_min', 'md_cmin_cand_max_std', 'md_cmin_cand_mean_max', 'md_cmin_cand_mean_mean', 'md_cmin_cand_mean_min', 'md_cmin_cand_mean_std', 'md_cmin_cand_min_max', 'md_cmin_cand_min_mean', 'md_cmin_cand_min_min', 'md_cmin_cand_min_std', 'md_cmin_cand_std_max', 'md_cmin_cand_std_mean', 'md_cmin_cand_std_min', 'md_cmin_cand_std_std', 'md_cmin_max', 'md_cmin_mean', 'md_cmin_min', 'md_cmin_std', 'md_d_cand_max_max', 'md_d_cand_max_mean', '

# Assessor Tests

Split data into train 60% and validation 40%. To prevent leakage, the data is binned in buckets by lat/long in 1km. The split is done by buckets ensuring no leakage. 

In [7]:
train_idx, test_idx = train_test_split(data.index, test_size=0.6, random_state=SEED)

In [8]:
X_train = data.loc[train_idx]
y_train = iou.loc[train_idx]
y_train_lrp = lrp.loc[train_idx]
conf_train = conf.loc[train_idx]
conf_train = pd.DataFrame(conf_train)

X_test = data.loc[test_idx]
y_test = iou.loc[test_idx]
y_test_lrp = lrp.loc[test_idx]
conf_test = conf.loc[test_idx]
conf_test = pd.DataFrame(conf_test)

print("X:", len(X_train), len(X_test))
print("y:", len(y_train), len(y_test))
print("Conf:", len(conf_train), len(conf_test))
print(X_train.columns)

X: 3866 5801
y: 3866 5801
Conf: 3866 5801
Index(['num_detections', 'md_N_min', 'md_N_max', 'md_N_mean', 'md_N_std',
       'md_rmin_min', 'md_rmin_max', 'md_rmin_mean', 'md_rmin_std',
       'md_rmax_min',
       ...
       'md_rdstd_min', 'md_rdstd_max', 'md_rdstd_mean', 'md_rdstd_std',
       'md_p_c0_mean', 'md_p_c0_max', 'md_p_c1_mean', 'md_p_c1_max',
       'md_p_c2_mean', 'md_p_c2_max'],
      dtype='object', length=191)


In [9]:
model_results = []

### IoU

#### Baseline

Random Prediction. Mean of metric over training set.

In [10]:
mean_iou_train = np.mean(y_train)
iou_pred_test = np.full_like(y_test, mean_iou_train)
record_regression_results(model_results, 'IoU', 'Constant Mean Predictor', y_test, iou_pred_test)


Test R2 score -0.0005
Test MAE score 0.1444
Test MSE score 0.0345
Test P95 score 0.3848
Test ECE score 0.0604


{'r2': -0.0004515334720234243,
 'mae': 0.14439896986761735,
 'mse': 0.034451811919881004,
 'p95': 0.38484031631568316,
 'ece': 0.06036912202835093}

#### Linear Regression

Train models with cv and then test.

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),    ],
    remainder='passthrough'
)
linear_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_grid = GridSearchCV(
    linear_reg_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_grid.fit(X_train, y_train)

best_idx = linear_reg_grid.best_index_
mean_train_r2 = linear_reg_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_grid.best_params_}")
best_linear_reg = linear_reg_grid.best_estimator_

Mean CV train r2 score 0.2887
Mean CV test r2 score 0.2172
Mean CV train MAE score 0.1203
Mean CV test MAE score 0.1262
Mean CV train MSE score 0.0242
Mean CV test MSE score 0.0265
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [12]:
y_pred_iou_lr = best_linear_reg.predict(X_test)
record_regression_results(model_results, 'IoU', 'Linear Regression', y_test, y_pred_iou_lr)


Test R2 score -0.1561
Test MAE score 0.1264
Test MSE score 0.0398
Test P95 score 0.3366
Test ECE score 0.0155


{'r2': -0.1561084790508429,
 'mae': 0.12637269606479418,
 'mse': 0.03981205540363453,
 'p95': 0.3365718957287436,
 'ece': 0.01547278872870352}

#### Decision Trees

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
    ],
    remainder='passthrough'
)
decision_tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_grid = GridSearchCV(
    decision_tree_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_grid.fit(X_train, y_train)

best_idx = decision_tree_grid.best_index_
mean_train_r2 = decision_tree_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_grid.best_params_}")
best_decision_tree = decision_tree_grid.best_estimator_


/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda3/envs/pred_ad/lib/py

Mean CV train r2 score 0.2896
Mean CV test r2 score 0.1881
Mean CV train MAE score 0.1193
Mean CV test MAE score 0.1268
Mean CV train MSE score 0.0241
Mean CV test MSE score 0.0274
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


In [14]:
y_pred_iou_dt = best_decision_tree.predict(X_test)
record_regression_results(model_results, 'IoU', 'Decision Tree', y_test, y_pred_iou_dt)


Test R2 score 0.1833
Test MAE score 0.1277
Test MSE score 0.0281
Test P95 score 0.3499
Test ECE score 0.0150


{'r2': 0.18334723900238004,
 'mae': 0.1276644003969714,
 'mse': 0.028122469089631622,
 'p95': 0.34988625623915576,
 'ece': 0.014966354189619578}

#### Random Forest

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
    ],
    remainder='passthrough'
)
rand_forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_grid = GridSearchCV(
    rand_forest_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_grid.fit(X_train, y_train)

best_idx = rand_forest_grid.best_index_
mean_train_r2 = rand_forest_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_grid.best_params_}")
best_rand_forest = rand_forest_grid.best_estimator_


/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda3/envs/pred_ad/lib/py

Mean CV train r2 score 0.4487
Mean CV test r2 score 0.2687
Mean CV train MAE score 0.1053
Mean CV test MAE score 0.1211
Mean CV train MSE score 0.0187
Mean CV test MSE score 0.0248
Best params: {'model__max_depth': 20, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 200}


In [16]:
y_pred_iou_rf = best_rand_forest.predict(X_test)
record_regression_results(model_results, 'IoU', 'Random Forest', y_test, y_pred_iou_rf)


Test R2 score 0.2735
Test MAE score 0.1207
Test MSE score 0.0250
Test P95 score 0.3245
Test ECE score 0.0073


{'r2': 0.2735085411596304,
 'mae': 0.12065515155563483,
 'mse': 0.025017650794643208,
 'p95': 0.32448803640458246,
 'ece': 0.007250782493116433}

#### MLP

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
    ],
    remainder='passthrough'
)
mlp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_grid = GridSearchCV(
    mlp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_grid.fit(X_train, y_train)

best_idx = mlp_grid.best_index_
mean_train_r2 = mlp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_grid.best_params_}")
best_mlp = mlp_grid.best_estimator_


Mean CV train r2 score 0.4954
Mean CV test r2 score 0.0086
Mean CV train MAE score 0.0995
Mean CV test MAE score 0.1395
Mean CV train MSE score 0.0171
Mean CV test MSE score 0.0334
Best params: {'model__activation': 'relu', 'model__alpha': 0.0001, 'model__hidden_layer_sizes': (50, 25), 'model__learning_rate_init': 0.01}


In [18]:
y_pred_iou_mlp = best_mlp.predict(X_test)
record_regression_results(model_results, 'IoU', 'MLP', y_test, y_pred_iou_mlp)


Test R2 score -0.5813
Test MAE score 0.1409
Test MSE score 0.0545
Test P95 score 0.3720
Test ECE score 0.0513


{'r2': -0.5813128341311433,
 'mae': 0.14094548788976477,
 'mse': 0.05445450431657876,
 'p95': 0.37198861610911615,
 'ece': 0.05128378425961953}

#### XGBoost

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
    ],
    remainder='passthrough'
)
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_grid.fit(X_train, y_train)


best_idx = xgb_grid.best_index_
mean_train_r2 = xgb_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_grid.best_params_}")
best_xgb = xgb_grid.best_estimator_

Mean CV train r2 score 0.5773
Mean CV test r2 score 0.2642
Mean CV train MAE score 0.0951
Mean CV test MAE score 0.1216
Mean CV train MSE score 0.0144
Mean CV test MSE score 0.0249
Best params: {'model__learning_rate': 0.01, 'model__n_estimators': 200}


In [20]:
y_pred_iou_xgb = best_xgb.predict(X_test)
record_regression_results(model_results, 'IoU', 'XGBoost', y_test, y_pred_iou_xgb)


Test R2 score 0.2709
Test MAE score 0.1213
Test MSE score 0.0251
Test P95 score 0.3237
Test ECE score 0.0104


{'r2': 0.27092037943682,
 'mae': 0.12134553051911927,
 'mse': 0.025106777411884773,
 'p95': 0.323717455069224,
 'ece': 0.010396681696508335}

#### Autogluon

##### Tabluar

In [21]:
train = pd.merge(X_train, y_train, left_index=True, right_index=True)

In [22]:
autoglue_model = TabularPredictor(label="iou", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/iou/")
autoglue_predictor = autoglue_model.fit(train)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #100-Ubuntu SMP PREEMPT_DYNAMIC Tue Jan 13 16:40:06 UTC 2026
CPU Count:          32
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 23.55/23.55 GB
Total GPU Memory:   Free: 23.55 GB, Allocated: 0.00 GB, Total: 23.55 GB
GPU Count:          1
Memory Avail:       17.29 GB / 30.46 GB (56.8%)
Disk Space Avail:   1861.49 GB / 3542.28 GB (52.6%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new

In [23]:
autoglue_predictor.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2   0.250469          r2       0.041650  6.187771                0.000216           0.015298            2       True          9
1           LightGBMXT   0.234499          r2       0.001057  0.742343                0.001057           0.742343            1       True          1
2        ExtraTreesMSE   0.226505          r2       0.024744  0.676996                0.024744           0.676996            1       True          5
3             CatBoost   0.220892          r2       0.001923  1.207662                0.001923           1.207662            1       True          4
4              XGBoost   0.197947          r2       0.002150  1.956172                0.002150           1.956172            1       True          6
5      RandomForestMSE   0.193584          r

/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestMSE': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesMSE': 'XTModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.23449884766638662,
  'LightGBM': 0.1867036172807568,
  'RandomForestMSE': 0.19358360086088144,
  'CatBoost': 0.22089153020294428,
  'ExtraTreesMSE': 0.22650505464361592,
  'XGBoost': 0.19794659316866914,
  'NeuralNetTorch': 0.19077968541044588,
  'LightGBMLarge': 0.17668190653754112,
  'WeightedEnsemble_L2': 0.25046915540817305},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'RandomForestMSE': ['RandomForestMSE'],
  'CatBoost': ['CatBoost'],
  'ExtraTreesMSE': ['ExtraTreesMSE'],
  'XGBoost': ['XGBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'LightGBMLarge': 

In [24]:
y_pred_iou_autg = autoglue_predictor.predict(X_test)
record_regression_results(model_results, 'IoU', 'Autogluon_Tab', y_test, y_pred_iou_autg)


Test R2 score 0.2990
Test MAE score 0.1184
Test MSE score 0.0241
Test P95 score 0.3197
Test ECE score 0.0068


{'r2': 0.29900423254480635,
 'mae': 0.11843665021371194,
 'mse': 0.024139674465973836,
 'p95': 0.319747163189782,
 'ece': 0.0068310889117156265}

##### Images

In [25]:
X_train_img = pd.merge(X_train, img_path, left_index=True, right_index=True)
X_test_img = pd.merge(X_test, img_path, how="left", right_index=True, left_index=True)
train_img = pd.merge(X_train_img, y_train, left_index=True, right_index=True)

In [26]:
#autoglue_model_img = MultiModalPredictor(label="iou", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/iou/img/")
hyperparams = {
    "env.strategy": "ddp_notebook",
    "env.num_gpus": 1,
    "env.num_workers": 0,
    #"optim.max_epochs": 15,
    #"optim.patience": 3,
    #"optim.learning_rate": 1e-4,
    #"optim.weight_decay": 1e-4,
    #"optim.warmup_steps": 500,
    #"env.per_gpu_batch_size": 8,
    #"env.batch_size": 128,
    #"model.timm_image.checkpoint_name": "convnext_base",
}
#autoglue_predictor_img = autoglue_model_img.fit(train_img, hyperparameters=hyperparams)

In [27]:
#autoglue_predictor_img.fit_summary(verbosity=2)

In [28]:
#y_pred = autoglue_predictor_img.predict(X_test_img)
#record_regression_results(model_results, 'IoU', 'Autogluon_Mult', y_test, y_pred)


### LRP


#### Baseline

Predict Only the Mean


In [29]:
mean_lrp_train = np.mean(y_train_lrp)
lrp_pred_test = np.full_like(y_test_lrp, mean_lrp_train)
record_regression_results(model_results, 'LRP', 'Constant Mean Predictor', y_test_lrp, lrp_pred_test)


Test R2 score -0.0008
Test MAE score 0.1321
Test MSE score 0.0287
Test P95 score 0.3751
Test ECE score 0.0145


{'r2': -0.0008094621802297031,
 'mae': 0.1320842776651714,
 'mse': 0.028747084490733734,
 'p95': 0.3750816030679314,
 'ece': 0.014548146724700894}

#### Linear Regression


Train models with cv and then test.


In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
    ],
    remainder='passthrough'
)
linear_reg_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
linear_reg_lrp_grid = GridSearchCV(
    linear_reg_lrp_pipeline,
    param_grid=LINEAR_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
linear_reg_lrp_grid.fit(X_train, y_train_lrp)

best_idx = linear_reg_lrp_grid.best_index_
mean_train_r2 = linear_reg_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = linear_reg_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -linear_reg_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -linear_reg_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -linear_reg_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -linear_reg_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {linear_reg_lrp_grid.best_params_}")
best_linear_reg_lrp = linear_reg_lrp_grid.best_estimator_


/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Mean CV train r2 score 0.3426
Mean CV test r2 score 0.2768
Mean CV train MAE score 0.1044
Mean CV test MAE score 0.1095
Mean CV train MSE score 0.0187
Mean CV test MSE score 0.0205
Best params: {'model__fit_intercept': True, 'model__positive': False}


In [31]:
y_pred_lrp_lr = best_linear_reg_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Linear Regression', y_test_lrp, y_pred_lrp_lr)


Test R2 score -0.1664
Test MAE score 0.1109
Test MSE score 0.0335
Test P95 score 0.2969
Test ECE score 0.0202


{'r2': -0.166382084923137,
 'mae': 0.11094037484056678,
 'mse': 0.03350296496070234,
 'p95': 0.29691568600440443,
 'ece': 0.02019799722410282}

#### Decision Trees


In [32]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
    ],
    remainder='passthrough'
)
decision_tree_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=SEED))
])
decision_tree_lrp_grid = GridSearchCV(
    decision_tree_lrp_pipeline,
    param_grid=TREE_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
decision_tree_lrp_grid.fit(X_train, y_train_lrp)

best_idx = decision_tree_lrp_grid.best_index_
mean_train_r2 = decision_tree_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = decision_tree_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -decision_tree_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -decision_tree_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -decision_tree_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -decision_tree_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {decision_tree_lrp_grid.best_params_}")
best_decision_tree_lrp = decision_tree_lrp_grid.best_estimator_


/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
180 fits failed out of a total of 360.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda3/envs/pred_ad/lib/py

Mean CV train r2 score 0.3303
Mean CV test r2 score 0.2219
Mean CV train MAE score 0.1056
Mean CV test MAE score 0.1133
Mean CV train MSE score 0.0190
Mean CV test MSE score 0.0220
Best params: {'model__max_depth': None, 'model__max_features': None, 'model__min_samples_leaf': 100, 'model__min_samples_split': 100}


In [33]:
y_pred_lrp_dt = best_decision_tree_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Decision Tree', y_test_lrp, y_pred_lrp_dt)


Test R2 score 0.2296
Test MAE score 0.1131
Test MSE score 0.0221
Test P95 score 0.2934
Test ECE score 0.0132


{'r2': 0.22956579674615385,
 'mae': 0.11313784500515403,
 'mse': 0.022129823880003438,
 'p95': 0.29341363172922563,
 'ece': 0.013207183073685288}

#### Random Forest


In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
    ],
    remainder='passthrough'
)
rand_forest_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])
rand_forest_lrp_grid = GridSearchCV(
    rand_forest_lrp_pipeline,
    param_grid=RF_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
rand_forest_lrp_grid.fit(X_train, y_train_lrp)

best_idx = rand_forest_lrp_grid.best_index_
mean_train_r2 = rand_forest_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = rand_forest_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -rand_forest_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -rand_forest_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -rand_forest_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -rand_forest_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {rand_forest_lrp_grid.best_params_}")
best_rand_forest_lrp = rand_forest_lrp_grid.best_estimator_


/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
360 fits failed out of a total of 720.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
360 fits failed with the following error:
Traceback (most recent call last):
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/felix/miniconda3/envs/pred_ad/lib/py

Mean CV train r2 score 0.4881
Mean CV test r2 score 0.3230
Mean CV train MAE score 0.0920
Mean CV test MAE score 0.1055
Mean CV train MSE score 0.0146
Mean CV test MSE score 0.0192
Best params: {'model__max_depth': 20, 'model__max_features': None, 'model__min_samples_leaf': 2, 'model__min_samples_split': 100, 'model__n_estimators': 100}


In [35]:
y_pred_lrp_rf = best_rand_forest_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Random Forest', y_test_lrp, y_pred_lrp_rf)


Test R2 score 0.3300
Test MAE score 0.1058
Test MSE score 0.0192
Test P95 score 0.2789
Test ECE score 0.0101


{'r2': 0.3300457244802796,
 'mae': 0.10582982848573573,
 'mse': 0.019243655152238588,
 'p95': 0.27885296308802565,
 'ece': 0.01008476921965844}

#### MLP


In [36]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
    ],
    remainder='passthrough'
)
mlp_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPRegressor(max_iter=500, random_state=SEED))
])
mlp_lrp_grid = GridSearchCV(
    mlp_lrp_pipeline,
    param_grid=MLP_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
mlp_lrp_grid.fit(X_train, y_train_lrp)

best_idx = mlp_lrp_grid.best_index_
mean_train_r2 = mlp_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = mlp_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -mlp_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -mlp_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -mlp_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -mlp_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {mlp_lrp_grid.best_params_}")
best_mlp_lrp = mlp_lrp_grid.best_estimator_


Mean CV train r2 score 0.6063
Mean CV test r2 score 0.0675
Mean CV train MAE score 0.0785
Mean CV test MAE score 0.1216
Mean CV train MSE score 0.0112
Mean CV test MSE score 0.0263
Best params: {'model__activation': 'relu', 'model__alpha': 0.01, 'model__hidden_layer_sizes': (100, 50), 'model__learning_rate_init': 0.01}


In [37]:
y_pred_lrp_mlp = best_mlp_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'MLP', y_test_lrp, y_pred_lrp_mlp)


Test R2 score -0.1008
Test MAE score 0.1236
Test MSE score 0.0316
Test P95 score 0.3353
Test ECE score 0.0363


{'r2': -0.10076790452986306,
 'mae': 0.12361339556995404,
 'mse': 0.03161827415907199,
 'p95': 0.3353434540565131,
 'ece': 0.03625710298431019}

#### XGBoost

In [38]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
    ],
    remainder='passthrough'
)
xgb_lrp_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor())
])
xgb_lrp_grid = GridSearchCV(
    xgb_lrp_pipeline,
    param_grid=XGB_PARAM_GRID,
    **GRID_SEARCH_KWARGS
)
xgb_lrp_grid.fit(X_train, y_train_lrp)


best_idx = xgb_lrp_grid.best_index_
mean_train_r2 = xgb_lrp_grid.cv_results_['mean_train_r2'][best_idx]
mean_test_r2 = xgb_lrp_grid.cv_results_['mean_test_r2'][best_idx]
mean_train_mae = -xgb_lrp_grid.cv_results_['mean_train_mae'][best_idx]
mean_test_mae = -xgb_lrp_grid.cv_results_['mean_test_mae'][best_idx]
mean_train_mse = -xgb_lrp_grid.cv_results_['mean_train_mse'][best_idx]
mean_test_mse = -xgb_lrp_grid.cv_results_['mean_test_mse'][best_idx]
print(f"Mean CV train r2 score {mean_train_r2:.4f}")
print(f"Mean CV test r2 score {mean_test_r2:.4f}")
print(f"Mean CV train MAE score {mean_train_mae:.4f}")
print(f"Mean CV test MAE score {mean_test_mae:.4f}")
print(f"Mean CV train MSE score {mean_train_mse:.4f}")
print(f"Mean CV test MSE score {mean_test_mse:.4f}")
print(f"Best params: {xgb_lrp_grid.best_params_}")
best_xgb_lrp = xgb_lrp_grid.best_estimator_

Mean CV train r2 score 0.7750
Mean CV test r2 score 0.3183
Mean CV train MAE score 0.0614
Mean CV test MAE score 0.1051
Mean CV train MSE score 0.0064
Mean CV test MSE score 0.0193
Best params: {'model__learning_rate': 0.1, 'model__n_estimators': 50}


In [39]:
y_pred_lrp_xgb = best_xgb_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'XGBoost', y_test_lrp, y_pred_lrp_xgb)


Test R2 score 0.3339
Test MAE score 0.1048
Test MSE score 0.0191
Test P95 score 0.2881
Test ECE score 0.0101


{'r2': 0.3339288368364387,
 'mae': 0.10482554284252395,
 'mse': 0.019132117278939174,
 'p95': 0.288107693195343,
 'ece': 0.010090301433856565}

#### Autogluon

##### Tabluar

In [40]:
train_lrp = pd.merge(X_train, y_train_lrp, left_index=True, right_index=True)

In [41]:
autoglue_model_lrp = TabularPredictor(label="lrp", problem_type="regression", eval_metric="r2", path="../models/assessors/autoglue_tab/lrp/tab/")
autoglue_predictor_lrp = autoglue_model_lrp.fit(train_lrp)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #100-Ubuntu SMP PREEMPT_DYNAMIC Tue Jan 13 16:40:06 UTC 2026
CPU Count:          32
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 23.55/23.55 GB
Total GPU Memory:   Free: 23.55 GB, Allocated: 0.00 GB, Total: 23.55 GB
GPU Count:          1
Memory Avail:       8.15 GB / 30.46 GB (26.8%)
Disk Space Avail:   1861.48 GB / 3542.28 GB (52.6%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new 

[1000]	valid_set's l2: 0.0219048	valid_set's r2: 0.240802


	0.2408	 = Validation score   (r2)
	5.17s	 = Training   runtime
	0.0s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ...
	Fitting 1 model on all data | Fitting with cpus=32, gpus=0, mem=0.0/8.1 GB
	Ensemble Weights: {'LightGBMXT': 0.529, 'NeuralNetTorch': 0.294, 'CatBoost': 0.118, 'XGBoost': 0.059}
	0.3005	 = Validation score   (r2)
	0.02s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 18.26s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 17789.1 rows/s (500 batch size)
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/home/felix/Predictability-AD/models/assessors/autoglue_tab/lrp/tab")


In [42]:
autoglue_predictor_lrp.fit_summary(verbosity=2)

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val eval_metric  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2   0.300525          r2       0.028107  7.450452                0.000236           0.015522            2       True          9
1           LightGBMXT   0.288019          r2       0.001052  0.713235                0.001052           0.713235            1       True          1
2             CatBoost   0.278159          r2       0.001683  2.347586                0.001683           2.347586            1       True          4
3             LightGBM   0.265703          r2       0.000771  0.646510                0.000771           0.646510            1       True          2
4              XGBoost   0.251763          r2       0.012193  2.045613                0.012193           2.045613            1       True          6
5        ExtraTreesMSE   0.248231          r

/home/felix/miniconda3/envs/pred_ad/lib/python3.11/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestMSE': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesMSE': 'XTModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.2880190280370567,
  'LightGBM': 0.26570290014926723,
  'RandomForestMSE': 0.23512892451195233,
  'CatBoost': 0.27815942731134446,
  'ExtraTreesMSE': 0.2482312444815714,
  'XGBoost': 0.2517625855109118,
  'NeuralNetTorch': 0.24711479842970718,
  'LightGBMLarge': 0.2408049741752344,
  'WeightedEnsemble_L2': 0.30052515649402733},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'RandomForestMSE': ['RandomForestMSE'],
  'CatBoost': ['CatBoost'],
  'ExtraTreesMSE': ['ExtraTreesMSE'],
  'XGBoost': ['XGBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'LightGBMLarge': ['L

In [43]:
y_pred_lrp_autg = autoglue_predictor_lrp.predict(X_test)
record_regression_results(model_results, 'LRP', 'Autogluon_Tab', y_test_lrp, y_pred_lrp_autg)


Test R2 score 0.3534
Test MAE score 0.1028
Test MSE score 0.0186
Test P95 score 0.2795
Test ECE score 0.0106


{'r2': 0.3534227279644576,
 'mae': 0.10277729024097562,
 'mse': 0.018572177993303792,
 'p95': 0.27951955795288086,
 'ece': 0.010553976996687696}

### Save predictions

# Final Model Comparison


In [44]:

results_df = pd.DataFrame(model_results, index=None)
results_df["test_r2"] = results_df["test_r2"].round(4)
results_df["test_mae"] = results_df["test_mae"].round(4)
#results_df["test_mse"] = results_df["test_mse"].round(4)
results_df.to_csv(f"../results/assessors/{model_det}_ass_results_table.csv")
display(results_df)


,target,model,test_r2,test_mae,test_ece
0,IoU,Constant Mean Predictor,-0.0005,0.1444,0.060369
1,IoU,Linear Regression,-0.1561,0.1264,0.015473
2,IoU,Decision Tree,0.1833,0.1277,0.014966
3,IoU,Random Forest,0.2735,0.1207,0.007251
4,IoU,MLP,-0.5813,0.1409,0.051284
5,IoU,XGBoost,0.2709,0.1213,0.010397
6,IoU,Autogluon_Tab,0.2990,0.1184,0.006831
7,LRP,Constant Mean Predictor,-0.0008,0.1321,0.014548
8,LRP,Linear Regression,-0.1664,0.1109,0.020198
9,LRP,Decision Tree,0.2296,0.1131,0.013207


In [45]:
print(results_df.to_latex(index=False))

\begin{tabular}{llrrr}
\toprule
target & model & test_r2 & test_mae & test_ece \\
\midrule
IoU & Constant Mean Predictor & -0.000500 & 0.144400 & 0.060369 \\
IoU & Linear Regression & -0.156100 & 0.126400 & 0.015473 \\
IoU & Decision Tree & 0.183300 & 0.127700 & 0.014966 \\
IoU & Random Forest & 0.273500 & 0.120700 & 0.007251 \\
IoU & MLP & -0.581300 & 0.140900 & 0.051284 \\
IoU & XGBoost & 0.270900 & 0.121300 & 0.010397 \\
IoU & Autogluon_Tab & 0.299000 & 0.118400 & 0.006831 \\
LRP & Constant Mean Predictor & -0.000800 & 0.132100 & 0.014548 \\
LRP & Linear Regression & -0.166400 & 0.110900 & 0.020198 \\
LRP & Decision Tree & 0.229600 & 0.113100 & 0.013207 \\
LRP & Random Forest & 0.330000 & 0.105800 & 0.010085 \\
LRP & MLP & -0.100800 & 0.123600 & 0.036257 \\
LRP & XGBoost & 0.333900 & 0.104800 & 0.010090 \\
LRP & Autogluon_Tab & 0.353400 & 0.102800 & 0.010554 \\
\bottomrule
\end{tabular}

